# Simplified Pipeline with Deterministic Figure Calculation

This notebook uses the **SimplifiedExtractor** which separates:

```
┌─────────────────────────────────────────────────────────────────────────┐
│                        OLD APPROACH (LLM does everything)               │
├─────────────────────────────────────────────────────────────────────────┤
│  Syllogism Text  →  LLM extracts types, terms, figure, mood, validity  │
│                     (figure errors cause wrong validity!)               │
└─────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────┐
│                        NEW APPROACH (Simplified + Deterministic)        │
├─────────────────────────────────────────────────────────────────────────┤
│  Syllogism Text  →  LLM extracts ONLY types + terms (NLU task)         │
│                           ↓                                             │
│                     Post-Processor computes figure deterministically    │
│                           ↓                                             │
│                     Validity Checker (24 Aristotelian forms)            │
└─────────────────────────────────────────────────────────────────────────┘
```

**Key improvements:**
1. LLM only does NLU (types + terms extraction)
2. Figure computed deterministically from term positions
3. I/O type confusion corrected by PropositionTypeValidator
4. All 24 Aristotelian forms recognized (including existential import)

**Known fixes:**
- `b9e979ff`: AAA-1 (was incorrectly AAA-3)
- `535b8c6e`: AEE-2 (was incorrectly AEE-3)
- `0752c43f`: IAI-4 (was incorrectly IAI-1)

## 1. Setup and Imports

In [1]:
import sys
import os
import json
from pathlib import Path
from datetime import datetime
from tqdm import tqdm

_nb_dir = Path.cwd() / "pamaldi_semeval_2026_11_task1" if (Path.cwd() / "pamaldi_semeval_2026_11_task1").exists() else Path.cwd()
sys.path.insert(0, str(_nb_dir))
import setup_path

from load_credentials import load_credentials_from_file
_creds = _nb_dir / "aws_credentials.txt" if (_nb_dir / "aws_credentials.txt").exists() else _nb_dir.parent / "aws_credentials.txt"
load_credentials_from_file(str(_creds))
print('✓ Credentials loaded')

from bedrock_client_bearer import BedrockClientBearer as BedrockClient
from neurosymbolic_pipeline import NeuroSymbolicPipeline
from simplified_extractor import SimplifiedExtractor
from evaluation import NeuroSymbolicEvaluator

print('✓ Imports successful')

✓ Credentials loaded
✓ Imports successful


## 2. Configuration

In [2]:
# ============================================================
# CONFIGURATION
# ============================================================

# Model
MODEL_ID = "qwen.qwen3-32b-v1:0"

# Number of samples
NUM_TRAIN = 10   # Training samples (-1 for all 1000)
NUM_TEST = -1     # Test samples (-1 for all 191)

# ============================================================
# SIMPLIFIED PIPELINE CONFIGURATION
# ============================================================
# Key change: use_simplified_extractor=True
# This uses deterministic figure calculation instead of LLM

PIPELINE_CONFIG = {
    "use_simplified_extractor": True,   # NEW: Use deterministic figure
    "use_reflexion": False,             # Not yet supported with simplified
    "use_self_consistency": True,
    "num_consistency_samples": 3,
    "temperature_schedule": [0.0, 0.3, 0.5],
    "use_fallback": True,
    "fallback_use_self_consistency": False
}

# Data paths
TRAIN_DATA_PATH = "data/train_data.json"
TEST_DATA_PATH = "data/test_data_subtask_1.json"
SIMPLIFIED_PROMPT_PATH = "prompts/structure_extraction_simplified.txt"
RESULTS_DIR = "../semeval_results"

# ============================================================
print(f"Model: {MODEL_ID}")
print(f"NUM_TRAIN: {NUM_TRAIN} {'(all)' if NUM_TRAIN == -1 else ''}")
print(f"NUM_TEST: {NUM_TEST} {'(all)' if NUM_TEST == -1 else ''}")
print(f"\nSimplified Pipeline:")
print(f"  Extractor: SimplifiedExtractor (deterministic figure)")
print(f"  Self-consistency: {PIPELINE_CONFIG['use_self_consistency']}")
if PIPELINE_CONFIG['use_self_consistency']:
    print(f"    Samples: {PIPELINE_CONFIG['num_consistency_samples']}")
print(f"  Fallback: {PIPELINE_CONFIG['use_fallback']}")

Model: qwen.qwen3-32b-v1:0
NUM_TRAIN: 10 
NUM_TEST: -1 (all)

Simplified Pipeline:
  Extractor: SimplifiedExtractor (deterministic figure)
  Self-consistency: True
    Samples: 3
  Fallback: True


## 3. Initialize Client

In [3]:
# Initialize Bedrock client
client = BedrockClient(model_id=MODEL_ID)
evaluator = NeuroSymbolicEvaluator()

print("✓ Client initialized")

✓ Client initialized


## 4. Load Data

In [4]:
# Load training data
with open(TRAIN_DATA_PATH, 'r', encoding='utf-8') as f:
    train_data = json.load(f)

if NUM_TRAIN > 0:
    train_data = train_data[:NUM_TRAIN]

print(f"Loaded {len(train_data)} training examples")

# Load simplified prompt
with open(SIMPLIFIED_PROMPT_PATH, 'r', encoding='utf-8') as f:
    simplified_prompt = f.read()

print(f"Simplified prompt loaded: {len(simplified_prompt)} chars")
print(f"\nPrompt preview (first 500 chars):")
print("-" * 40)
print(simplified_prompt[:500])
print("...")

Loaded 10 training examples
Simplified prompt loaded: 6545 chars

Prompt preview (first 500 chars):
----------------------------------------
You are a formal logic expert. Your task is to extract the logical structure from syllogisms.

## YOUR ONLY JOB: Extract Types and Terms

You need to identify for each proposition:
1. **Type** (A, E, I, or O)
2. **Subject** (the term before the copula)
3. **Predicate** (the term after the copula)

**DO NOT** calculate figure, mood, middle term, or validity. That will be computed automatically.

---

## Proposition Types

| Type | Pattern | Example |
|------|---------|---------|
| **A** | "All S 
...


## 5. TRAINING PHASE with Simplified Extractor

The simplified extractor:
1. Asks LLM for types (A/E/I/O) and terms (subject/predicate) only
2. Applies I/O type correction (PropositionTypeValidator)
3. Computes figure deterministically from term positions
4. Checks validity against 24 Aristotelian forms

In [5]:
# Create output directory
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = os.path.join(RESULTS_DIR, f"simplified_train_{timestamp}")
os.makedirs(output_dir, exist_ok=True)

# Initialize pipeline with simplified extractor
pipeline = NeuroSymbolicPipeline(
    bedrock_client=client,
    prompt_path=SIMPLIFIED_PROMPT_PATH,
    results_dir=output_dir,
    run_name="train",
    verbose=False,
    **PIPELINE_CONFIG
)

print(f"\n✓ Pipeline initialized with SimplifiedExtractor")
print(f"  Output dir: {output_dir}")

  System prompt saved to: ../semeval_results\simplified_train_20260301_143928\neurosymbolic_train_20260301_143928\system_prompt.txt
  Results dir: ../semeval_results\simplified_train_20260301_143928\neurosymbolic_train_20260301_143928
✓ NeuroSymbolic Pipeline initialized
  Extractor: Simplified (deterministic figure)
  Use reflexion: False
  Use self-consistency: True
    Samples: 3, Temperatures: [0.0, 0.3, 0.5]
  Verify figure: False
  Use fallback: True
    Fallback self-consistency: False
  Results dir: ../semeval_results\simplified_train_20260301_143928\neurosymbolic_train_20260301_143928

✓ Pipeline initialized with SimplifiedExtractor
  Output dir: ../semeval_results\simplified_train_20260301_143928


In [6]:
# Run training
print(f"\nStarting training with simplified extractor...")
print(f"Processing {len(train_data)} samples")
print("-" * 60)

train_results = pipeline.run(train_data, verbose=True)

print("\n✓ Training complete!")


Starting training with simplified extractor...
Processing 10 samples
------------------------------------------------------------


Processing: 100%|██████████| 10/10 [00:49<00:00,  4.91s/it]


✓ Summary saved to: ../semeval_results\simplified_train_20260301_143928\neurosymbolic_train_20260301_143928
  Methods: 10 symbolic, 0 fallback, 0 failed

  ┌─────────────────────────────────────┐
  │  COMBINED SCORE:   100.00           │
  │  Accuracy:         100.00%          │
  │  Content Effect:     0.00           │
  └─────────────────────────────────────┘

✓ Training complete!


## 6. Evaluate Training Results

In [7]:
# Evaluate training results
train_metrics = evaluator.evaluate(train_results)
evaluator.print_report(train_metrics)

# Official evaluation (Training)
from pathlib import Path
if train_results and len(train_results) > 0:
    print("\n" + "=" * 80)
    print("OFFICIAL EVALUATION — TRAINING (Task 1 & 3)")
    print("=" * 80)
    reference_file = os.path.join(output_dir, 'reference.json')
    predictions_file = os.path.join(output_dir, 'predictions_official.json')
    results_file = os.path.join(output_dir, 'official_evaluation_results.json')
    reference_data = [{'id': item['id'], 'validity': item.get('validity', False), 'plausibility': item.get('plausibility', False)} for item in train_data]
    with open(reference_file, 'w', encoding='utf-8') as f:
        json.dump(reference_data, f, indent=2)
    predictions_official = [{'id': r['id'], 'validity': (r.get('prediction') == 'VALID')} for r in train_results]
    with open(predictions_file, 'w', encoding='utf-8') as f:
        json.dump(predictions_official, f, indent=2)
    _eval_kit = Path.cwd().parent / "evaluation_kit" / "task 1 & 3"
    if _eval_kit.exists() and str(_eval_kit) not in sys.path:
        sys.path.insert(0, str(_eval_kit))
    from evaluation_script import run_full_scoring
    run_full_scoring(reference_file, predictions_file, results_file)
    if os.path.exists(results_file):
        with open(results_file, 'r') as f:
            eval_results = json.load(f)
        print(f"Accuracy: {eval_results.get('accuracy', 0):.2f}%  Content Effect: {eval_results.get('content_effect', 0):.2f}%  Combined: {eval_results.get('combined_score', 0):.2f}%")
        print("=" * 80)


NEURO-SYMBOLIC EVALUATION REPORT

Total Items: 10
Processed: 10
Extraction Rate: 100.0%

Overall Accuracy: 100.00%
  Correct: 10
  Incorrect: 0

Content Effect: 0.00%
  Plausible Accuracy: 100.00% (n=6)
  Implausible Accuracy: 100.00% (n=4)

By Ground Truth:
  VALID accuracy: 100.00% (n=3)
  INVALID accuracy: 100.00% (n=7)


OFFICIAL EVALUATION — TRAINING (Task 1 & 3)
Scoring complete. Results written to ../semeval_results\simplified_train_20260301_143928\official_evaluation_results.json
Accuracy: 100.00%  Content Effect: 0.00%  Combined: 100.00%


## 7. Analyze Errors

Check if the known figure errors are fixed.

In [8]:
# Check for known error cases
known_error_ids = ["b9e979ff", "535b8c6e", "0752c43f"]

print("KNOWN ERROR CASES CHECK")
print("=" * 60)

for result in train_results:
    rid = result.get("id", "")
    # Check if this ID starts with any known error ID
    for known_id in known_error_ids:
        if rid.startswith(known_id):
            gt = result.get("ground_truth", "N/A")
            pred = result.get("prediction", "N/A")
            details = result.get("validity_details", {})
            form = details.get("form", "N/A")
            figure = details.get("figure", "N/A")
            
            status = "✓ FIXED" if pred == gt else "✗ STILL WRONG"
            
            print(f"\nID: {rid}")
            print(f"  Ground Truth: {gt}")
            print(f"  Prediction:   {pred}")
            print(f"  Form:         {form}")
            print(f"  Figure:       {figure}")
            print(f"  Status:       {status}")

print("\n" + "=" * 60)

KNOWN ERROR CASES CHECK



In [9]:
# Show all errors
errors = [r for r in train_results if r.get("prediction") != r.get("ground_truth")]

print(f"ERRORS: {len(errors)} / {len(train_results)}")
print("=" * 60)

for i, err in enumerate(errors[:10]):  # Show first 10
    print(f"\n{i+1}. ID: {err.get('id', 'N/A')}")
    print(f"   GT: {err.get('ground_truth')} | Pred: {err.get('prediction')}")
    print(f"   Method: {err.get('method')}")
    if err.get('validity_details'):
        details = err['validity_details']
        print(f"   Form: {details.get('form')} | Figure: {details.get('figure')}")
    if err.get('error'):
        print(f"   Error: {err.get('error')[:100]}...")

if len(errors) > 10:
    print(f"\n... and {len(errors) - 10} more errors")

ERRORS: 0 / 10


## 8. TEST PHASE

In [10]:
test_results = None

if NUM_TEST == 0:
    print("⏭ Skipping test phase (NUM_TEST = 0)")
else:
    # Load test data
    with open(TEST_DATA_PATH, 'r', encoding='utf-8') as f:
        test_data = json.load(f)
    
    if NUM_TEST > 0:
        test_data = test_data[:NUM_TEST]
    
    print(f"Loaded {len(test_data)} test examples")
    
    # Initialize test pipeline with simplified extractor
    test_pipeline = NeuroSymbolicPipeline(
        bedrock_client=client,
        prompt_path=SIMPLIFIED_PROMPT_PATH,
        results_dir=RESULTS_DIR,
        run_name="test_simplified",
        verbose=False,
        **PIPELINE_CONFIG
    )
    
    # Run test
    print(f"\nRunning test with simplified extractor...")
    test_results = test_pipeline.run(test_data, verbose=True)
    
    # Save predictions
    test_pipeline.save_predictions(test_results)
    print("\n✓ Predictions saved!")

    # Official evaluation on test (Task 1 & 3)
    print("\n" + "=" * 80)
    print("OFFICIAL EVALUATION — TEST (Task 1 & 3)")
    print("=" * 80)
    test_output_dir = test_pipeline.results_dir
    reference_file = os.path.join(test_output_dir, 'reference.json')
    predictions_file = os.path.join(test_output_dir, 'predictions_official.json')
    results_file = os.path.join(test_output_dir, 'official_evaluation_results.json')
    reference_data = [{'id': item['id'], 'validity': item.get('validity', False), 'plausibility': item.get('plausibility', False)} for item in test_data]
    with open(reference_file, 'w', encoding='utf-8') as f:
        json.dump(reference_data, f, indent=2)
    predictions_official = [{'id': r['id'], 'validity': (r.get('prediction') == 'VALID')} for r in test_results]
    with open(predictions_file, 'w', encoding='utf-8') as f:
        json.dump(predictions_official, f, indent=2)
    _eval_kit = Path.cwd().parent / "evaluation_kit" / "task 1 & 3"
    if _eval_kit.exists() and str(_eval_kit) not in sys.path:
        sys.path.insert(0, str(_eval_kit))
    from evaluation_script import run_full_scoring
    run_full_scoring(reference_file, predictions_file, results_file)
    if os.path.exists(results_file):
        with open(results_file, 'r') as f:
            eval_results = json.load(f)
        print(f"Accuracy: {eval_results.get('accuracy', 0):.2f}%  Content Effect: {eval_results.get('content_effect', 0):.2f}%  Combined: {eval_results.get('combined_score', 0):.2f}%")
        print("=" * 80)

Loaded 191 test examples
  System prompt saved to: ../semeval_results\neurosymbolic_test_simplified_20260301_144018\system_prompt.txt
  Results dir: ../semeval_results\neurosymbolic_test_simplified_20260301_144018
✓ NeuroSymbolic Pipeline initialized
  Extractor: Simplified (deterministic figure)
  Use reflexion: False
  Use self-consistency: True
    Samples: 3, Temperatures: [0.0, 0.3, 0.5]
  Verify figure: False
  Use fallback: True
    Fallback self-consistency: False
  Results dir: ../semeval_results\neurosymbolic_test_simplified_20260301_144018

Running test with simplified extractor...


Processing:   0%|          | 0/191 [00:00<?, ?it/s]  ✗ No middle term found!
  ✗ Could not identify middle term!
Processing:  72%|███████▏  | 137/191 [11:55<04:56,  5.48s/it]  ✗ No middle term found!
  ✗ Could not identify middle term!
Processing:  89%|████████▉ | 170/191 [15:02<02:15,  6.48s/it]  ✗ No middle term found!
  ✗ Could not identify middle term!
Processing: 100%|██████████| 191/191 [17:01<00:00,  5.35s/it]


✓ Summary saved to: ../semeval_results\neurosymbolic_test_simplified_20260301_144018
  Methods: 184 symbolic, 7 fallback, 0 failed

  ┌─────────────────────────────────────┐
  │  COMBINED SCORE:    45.81           │
  │  Accuracy:          97.38%          │
  │  Content Effect:     2.08           │
  └─────────────────────────────────────┘

  Accuracy breakdown: Symbolic 97.83%, Fallback 85.71%

  Failures logged: 5 (see debug_failures.txt)
✓ Predictions saved to: ../semeval_results\neurosymbolic_test_simplified_20260301_144018\predictions.json (official format: validity=boolean)

✓ Predictions saved!

OFFICIAL EVALUATION — TEST (Task 1 & 3)
Scoring complete. Results written to ../semeval_results\neurosymbolic_test_simplified_20260301_144018\official_evaluation_results.json
Accuracy: 97.38%  Content Effect: 2.08%  Combined: 45.81%
